# ACA Final — Visualizaciones y modelo de segmentación
**Fundamentos de Inteligencia de Negocios · Especialización en Analítica Avanzada de Datos**

Este notebook reproduce, de principio a fin, la evidencia del documento ACA2:

1. Auditoría de calidad de datos y depuración trazable.
2. Construcción de variables RFM a nivel de cliente.
3. Selección del número de clusters y entrenamiento de K-Means.
4. Las tres visualizaciones finales: **comparación**, **tendencia** y **composición**.
5. Serialización del modelo para la aplicación Streamlit.

Todas las figuras se exportan en PNG a la carpeta `salidas/figuras/`.

## 0. Configuración

Cambia únicamente la variable `RUTA_DATASET` si el archivo está en otra ubicación.

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================
from pathlib import Path

RUTA_DATASET = Path("dataset_crudo_aca1.xlsx")   # <-- ajusta aquí si es necesario
CARPETA_SALIDA = Path("salidas")
CARPETA_FIGURAS = CARPETA_SALIDA / "figuras"
CARPETA_MODELO = CARPETA_SALIDA / "modelo"

RANGO_K = range(2, 7)        # valores de k a evaluar
SEMILLA = 42                 # reproducibilidad
DPI = 190                    # resolución de exportación

for carpeta in (CARPETA_FIGURAS, CARPETA_MODELO):
    carpeta.mkdir(parents=True, exist_ok=True)

print("Dataset :", RUTA_DATASET.resolve())
print("Salidas :", CARPETA_SALIDA.resolve())

In [ ]:
# ============================================================
# LIBRERÍAS
# ============================================================
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import FancyBboxPatch
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

# --- Identidad visual del informe -----------------------------------------
AZUL_OSCURO = "#1F4E78"
AZUL_MEDIO = "#2E75B6"
AZUL_CLARO = "#5B9BD5"
NARANJA = "#ED7D31"
GRIS = "#A5A5A5"
TINTA = "#17365D"
PALETA = [AZUL_OSCURO, NARANJA, AZUL_CLARO, "#70AD47", GRIS, AZUL_MEDIO]

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#D0D5DB",
    "axes.labelcolor": "#3C4450",
    "text.color": "#3C4450",
    "xtick.color": "#5A6470",
    "ytick.color": "#5A6470",
    "font.size": 10,
    "axes.titlesize": 12.5,
    "axes.titleweight": "bold",
    "axes.grid": False,
})


def guardar(fig, nombre):
    """Exporta la figura en PNG dentro de salidas/figuras/."""
    ruta = CARPETA_FIGURAS / f"{nombre}.png"
    fig.savefig(ruta, dpi=DPI, bbox_inches="tight", facecolor="white")
    print("Figura guardada:", ruta)
    return ruta


print("Librerías cargadas.")

## 1. Carga y perfil inicial

El archivo crudo se conserva intacto: todas las transformaciones se aplican sobre copias.

In [ ]:
# ============================================================
# CARGA DEL DATASET CRUDO
# ============================================================
crudo = pd.read_excel(RUTA_DATASET, engine="openpyxl")

print(f"Filas   : {len(crudo)}")
print(f"Columnas: {crudo.shape[1]}")
crudo.head()

In [ ]:
# ============================================================
# PERFIL INICIAL DE CALIDAD
# ============================================================
monto_esperado = (crudo["Cantidad"] * crudo["Precio_Unitario"]).round(2)

falla_aritmetica = (
    (crudo["Monto_Total"].round(2) != monto_esperado)
    | (crudo["Cantidad"] <= 0)
    | (crudo["Precio_Unitario"] <= 0)
    | (crudo["Monto_Total"] <= 0)
)

fechas_crudas = pd.to_datetime(
    crudo["Fecha_Compra"], format="mixed", dayfirst=True, errors="coerce"
)

perfil = {
    "Filas de entrada": len(crudo),
    "Columnas": crudo.shape[1],
    "Transacciones únicas": crudo["ID_Transaccion"].nunique(),
    "Duplicados exactos": int(crudo.duplicated().sum()),
    "ID de cliente nulos": int(crudo["ID_Cliente"].isna().sum()),
    "Categorías nulas": int(crudo["Categoria"].isna().sum()),
    "Fallas aritméticas o de signo": int(falla_aritmetica.sum()),
    "Fechas no convertibles": int(fechas_crudas.isna().sum()),
}

for clave, valor in perfil.items():
    print(f"{clave:<32} {valor}")

## 2. Depuración trazable

Tres reglas, cada una declarada y contabilizada:

| Regla | Efecto |
|---|---|
| Eliminar duplicados exactos | Evita inflar la frecuencia de compra |
| Imputar categoría desde `ID_Producto` | Recupera valores sin ambigüedad |
| Excluir registros sin cliente o sin coherencia aritmética | Protege el principio GIGO |

In [ ]:
# ============================================================
# DEPURACIÓN
# ============================================================
limpio = crudo.drop_duplicates().copy()

# Imputación de categoría a partir del producto (relación estable en el archivo)
mapa_categoria = (
    limpio.dropna(subset=["Categoria"])
    .groupby("ID_Producto")["Categoria"]
    .agg(lambda s: s.mode().iat[0])
    .to_dict()
)
limpio["Categoria"] = limpio["Categoria"].fillna(limpio["ID_Producto"].map(mapa_categoria))

# Estandarización de fechas a tipo datetime
limpio["Fecha_Compra"] = pd.to_datetime(
    limpio["Fecha_Compra"], format="mixed", dayfirst=True, errors="coerce"
)

# Bandera de calidad: qué registros son aptos para modelar
limpio["Monto_Esperado"] = (limpio["Cantidad"] * limpio["Precio_Unitario"]).round(2)
limpio["Apto_Modelo"] = (
    limpio["ID_Cliente"].notna()
    & limpio["Fecha_Compra"].notna()
    & (limpio["Cantidad"] > 0)
    & (limpio["Precio_Unitario"] > 0)
    & (limpio["Monto_Total"] > 0)
    & (limpio["Monto_Total"].round(2) == limpio["Monto_Esperado"])
)

transacciones = limpio.loc[limpio["Apto_Modelo"]].copy()
transacciones["Mes"] = transacciones["Fecha_Compra"].dt.to_period("M").astype(str)

embudo = [
    ("Dataset crudo", len(crudo), "Fuente original conservada sin modificar"),
    ("Sin duplicados exactos", len(limpio),
     f"Se retiran {len(crudo) - len(limpio)} filas duplicadas exactas"),
    ("Transacciones aptas", len(transacciones),
     f"Se excluyen {len(limpio) - len(transacciones)} registros sin cliente "
     f"o sin coherencia aritmética"),
]

for etapa, filas, nota in embudo:
    print(f"{etapa:<26} {filas:>4} filas   |  {nota}")

print(f"\nPeriodo válido: {transacciones['Fecha_Compra'].min().date()} "
      f"a {transacciones['Fecha_Compra'].max().date()}")

### Figura 1 — Trazabilidad de la depuración

Embudo que muestra cuántas filas sobreviven a cada regla. Es la evidencia visual que
acompaña la sección 3.2 del documento.

In [ ]:
# ============================================================
# FIGURA 1 — EMBUDO DE DEPURACIÓN
# ============================================================
fig, ax = plt.subplots(figsize=(8.6, 3.5))
ax.set_xlim(0, 100)
ax.set_ylim(0, len(embudo))
ax.axis("off")

maximo = embudo[0][1]
colores_embudo = [AZUL_OSCURO, AZUL_MEDIO, AZUL_CLARO]

for i, ((etapa, filas, nota), color) in enumerate(zip(embudo, colores_embudo)):
    y = len(embudo) - 1 - i
    ancho = filas / maximo * 62
    ax.add_patch(FancyBboxPatch(
        (0, y + 0.18), ancho, 0.58,
        boxstyle="round,pad=0.006,rounding_size=0.06",
        fc=color, ec="none",
    ))
    ax.text(1.6, y + 0.47, etapa, va="center", ha="left",
            color="white", fontsize=10.5, fontweight="bold")
    ax.text(ancho + 1.6, y + 0.47, f"{filas} filas", va="center", ha="left",
            color=TINTA, fontsize=10.5, fontweight="bold")
    ax.text(1.6, y + 0.03, nota, va="center", ha="left",
            color="#5A6470", fontsize=8.4)

ax.text(0, len(embudo) - 0.08, "Trazabilidad de la depuración",
        fontsize=12.5, fontweight="bold", color=TINTA, va="top")

fig.tight_layout()
guardar(fig, "fig1_embudo_depuracion")
plt.show()

## 3. Construcción de variables RFM

Se cambia la unidad de análisis: de transacción a cliente. Sin este paso, el algoritmo
describiría tipos de compra y no tipos de cliente.

La **recencia** se mide contra la fecha máxima del dataset, no contra la fecha real, para
no penalizar a todos los clientes por el desfase del archivo.

In [ ]:
# ============================================================
# AGREGACIÓN A NIVEL DE CLIENTE
# ============================================================
fecha_corte = transacciones["Fecha_Compra"].max()

rfm = (
    transacciones.groupby("ID_Cliente")
    .agg(
        Recencia=("Fecha_Compra", lambda s: (fecha_corte - s.max()).days),
        Frecuencia=("ID_Transaccion", "nunique"),
        Valor_Monetario=("Monto_Total", "sum"),
        Ticket_Promedio=("Monto_Total", "mean"),
    )
    .reset_index()
)

VARIABLES = ["Recencia", "Frecuencia", "Valor_Monetario", "Ticket_Promedio"]

print(f"Clientes modelados: {len(rfm)}")
print(f"Fecha de corte    : {fecha_corte.date()}\n")
rfm.describe().round(2)

## 4. Selección del número de clusters

Se estandarizan las variables para que la escala monetaria no domine las distancias y se
evalúa el rango de `k` con el coeficiente Silhouette.

In [ ]:
# ============================================================
# ESTANDARIZACIÓN Y EVALUACIÓN DE k
# ============================================================
escalador = StandardScaler()
X = escalador.fit_transform(rfm[VARIABLES])

silhouettes = {}
for k in RANGO_K:
    modelo_k = KMeans(n_clusters=k, random_state=SEMILLA, n_init=20).fit(X)
    silhouettes[k] = silhouette_score(X, modelo_k.labels_)
    print(f"k = {k}  ->  Silhouette = {silhouettes[k]:.4f}")

K_OPTIMO = max(silhouettes, key=silhouettes.get)
print(f"\nk seleccionado: {K_OPTIMO} (Silhouette = {silhouettes[K_OPTIMO]:.4f})")

### Figura 2 — Evaluación del número de clusters

In [ ]:
# ============================================================
# FIGURA 2 — CURVA SILHOUETTE
# ============================================================
fig, ax = plt.subplots(figsize=(7.6, 3.9))

ks = list(silhouettes)
valores = [silhouettes[k] for k in ks]

ax.plot(ks, valores, marker="o", lw=2.2, color=AZUL_OSCURO, zorder=2)
ax.scatter([K_OPTIMO], [silhouettes[K_OPTIMO]], s=170, color=NARANJA,
           zorder=3, edgecolor="white", linewidth=2)
ax.annotate(f"k = {K_OPTIMO}  ({silhouettes[K_OPTIMO]:.3f})",
            xy=(K_OPTIMO, silhouettes[K_OPTIMO]),
            xytext=(14, -4), textcoords="offset points",
            fontsize=9.5, fontweight="bold", color=NARANJA)

ax.set_title("¿Cuántos segmentos separan mejor a los clientes?", loc="left", pad=14)
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Coeficiente Silhouette")
ax.set_xticks(ks)
margen = (max(valores) - min(valores)) or 0.1
ax.set_ylim(min(valores) - margen * 0.25, max(valores) + margen * 0.35)
ax.grid(axis="y", alpha=0.25)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
guardar(fig, "fig2_silhouette")
plt.show()

## 5. Entrenamiento y perfilado de segmentos

Las etiquetas de negocio se derivan de la posición relativa de cada cluster frente a la
mediana del conjunto, no se asignan a mano. Así el código sigue siendo válido si el
dataset cambia.

In [ ]:
# ============================================================
# K-MEANS Y ETIQUETAS DE NEGOCIO
# ============================================================
modelo = KMeans(n_clusters=K_OPTIMO, random_state=SEMILLA, n_init=20).fit(X)
rfm["Cluster"] = modelo.labels_

resumen = (
    rfm.groupby("Cluster")
    .agg(
        Clientes=("ID_Cliente", "count"),
        Recencia_media=("Recencia", "mean"),
        Frecuencia_media=("Frecuencia", "mean"),
        Valor_medio=("Valor_Monetario", "mean"),
        Valor_total=("Valor_Monetario", "sum"),
    )
    .reset_index()
)

med_valor = resumen["Valor_medio"].median()
med_frec = resumen["Frecuencia_media"].median()
med_rec = resumen["Recencia_media"].median()


def etiquetar(fila):
    if fila.Valor_medio >= med_valor and fila.Frecuencia_media >= med_frec:
        return "Clientes de alto valor y alta frecuencia"
    if fila.Valor_medio >= med_valor:
        return "Clientes de alto valor y menor frecuencia"
    if fila.Recencia_media <= med_rec:
        return "Clientes recientes de valor moderado"
    return "Clientes de menor actividad y valor"


etiquetas = {}
for _, fila in resumen.iterrows():
    nombre = etiquetar(fila)
    if nombre in etiquetas.values():          # evita nombres repetidos
        nombre = f"{nombre} (segmento {int(fila.Cluster)})"
    etiquetas[int(fila.Cluster)] = nombre

resumen["Segmento"] = resumen["Cluster"].map(etiquetas)
resumen["Pct_Ingresos"] = resumen["Valor_total"] / resumen["Valor_total"].sum()
rfm["Segmento"] = rfm["Cluster"].map(etiquetas)

transacciones = transacciones.merge(
    rfm[["ID_Cliente", "Cluster", "Segmento"]], on="ID_Cliente", how="left"
)

resumen.sort_values("Valor_total", ascending=False).round(2)

## 6. Las tres visualizaciones finales

Cada figura responde **una** pregunta de negocio y aplica atributos preatentivos: el color
saturado se reserva para el dato que debe leerse primero, el resto se atenúa en gris.

### Figura 3 — Comparación: ¿qué segmento aporta más ingresos?

Barras horizontales ordenadas. Solo el segmento líder se destaca en naranja; los demás
quedan en gris para que la jerarquía sea inmediata.

In [ ]:
# ============================================================
# FIGURA 3 — COMPARACIÓN DE INGRESOS POR SEGMENTO
# ============================================================
datos = resumen.sort_values("Valor_total")

fig, ax = plt.subplots(figsize=(8.4, 0.95 * len(datos) + 2.2))

colores = [GRIS] * len(datos)
colores[-1] = NARANJA                      # atributo preatentivo: color

barras = ax.barh(datos["Segmento"], datos["Valor_total"], color=colores,
                 height=0.58, edgecolor="none")

tope = datos["Valor_total"].max()
for barra, valor, pct in zip(barras, datos["Valor_total"], datos["Pct_Ingresos"]):
    ax.text(valor + tope * 0.015, barra.get_y() + barra.get_height() / 2,
            f"{valor:,.0f}   ({pct:.0%})", va="center", ha="left",
            fontsize=10, fontweight="bold", color=TINTA)

ax.set_title("¿Qué segmento concentra los ingresos?", loc="left", pad=16)
ax.set_xlabel("Monto total del periodo analizado")
ax.set_xlim(0, tope * 1.28)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
ax.grid(axis="x", alpha=0.18)
ax.set_axisbelow(True)

fig.tight_layout()
guardar(fig, "fig3_comparacion_ingresos")
plt.show()

### Figura 4 — Tendencia: ¿cómo evoluciona el ingreso en el tiempo?

Líneas por segmento con etiqueta al final de cada serie, para no depender de una leyenda
lateral. El periodo disponible es corto, así que la lectura es comparativa, no estacional.

In [ ]:
# ============================================================
# FIGURA 4 — TENDENCIA MENSUAL POR SEGMENTO
# ============================================================
tendencia = (
    transacciones.groupby(["Mes", "Segmento"])["Monto_Total"]
    .sum()
    .reset_index()
    .sort_values("Mes")
)

orden = resumen.sort_values("Valor_total", ascending=False)["Segmento"].tolist()

fig, ax = plt.subplots(figsize=(8.6, 4.4))

for i, segmento in enumerate(orden):
    serie = tendencia[tendencia["Segmento"] == segmento]
    color = NARANJA if i == 0 else PALETA[(i + 2) % len(PALETA)]
    grosor = 2.8 if i == 0 else 1.8
    ax.plot(serie["Mes"], serie["Monto_Total"], marker="o", markersize=7,
            lw=grosor, color=color, label=segmento, zorder=3 - i * 0.1)
    if len(serie):
        ultimo = serie.iloc[-1]
        ax.annotate(f"{ultimo['Monto_Total']:,.0f}",
                    xy=(ultimo["Mes"], ultimo["Monto_Total"]),
                    xytext=(9, 0), textcoords="offset points",
                    va="center", fontsize=9, fontweight="bold", color=color)

ax.set_title("Evolución mensual de los ingresos por segmento", loc="left", pad=16)
ax.set_xlabel("Mes")
ax.set_ylabel("Monto total")
ax.margins(x=0.16)
ax.grid(axis="y", alpha=0.22)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8.6, loc="upper left", bbox_to_anchor=(0, -0.16), ncol=2)

fig.tight_layout()
guardar(fig, "fig4_tendencia_mensual")
plt.show()

### Figura 5 — Composición: ¿cómo se reparte la base de clientes?

Dona con el conteo total en el centro. Leída junto a la Figura 3, muestra si el tamaño de
un segmento se corresponde o no con su aporte económico.

In [ ]:
# ============================================================
# FIGURA 5 — COMPOSICIÓN DE LA BASE DE CLIENTES
# ============================================================
datos = resumen.sort_values("Clientes", ascending=False)

fig, ax = plt.subplots(figsize=(8.0, 4.6))

colores = [NARANJA if s == orden[0] else PALETA[(i + 2) % len(PALETA)]
           for i, s in enumerate(datos["Segmento"])]

cunas, _, textos = ax.pie(
    datos["Clientes"],
    colors=colores,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.0f}%\n({round(p * datos['Clientes'].sum() / 100)})",
    pctdistance=0.76,
    wedgeprops=dict(width=0.42, edgecolor="white", linewidth=2.5),
    textprops=dict(fontsize=9.5, fontweight="bold", color="white"),
)

ax.text(0, 0.08, f"{int(datos['Clientes'].sum())}", ha="center", va="center",
        fontsize=23, fontweight="bold", color=TINTA)
ax.text(0, -0.16, "clientes", ha="center", va="center", fontsize=10, color="#5A6470")

ax.set_title("Distribución de la base de clientes", loc="left", pad=16)
ax.legend(cunas, datos["Segmento"], frameon=False, fontsize=9,
          loc="center left", bbox_to_anchor=(1.0, 0.5))

fig.tight_layout()
guardar(fig, "fig5_composicion_clientes")
plt.show()

## 7. Serialización del modelo

Los tres artefactos deben viajar juntos a la aplicación Streamlit: sin el escalador, las
entradas del usuario se transformarían con una escala distinta a la del entrenamiento y la
predicción sería incorrecta.

In [ ]:
# ============================================================
# EXPORTACIÓN DE ARTEFACTOS
# ============================================================
joblib.dump(modelo, CARPETA_MODELO / "modelo_kmeans.pkl")
joblib.dump(escalador, CARPETA_MODELO / "scaler.pkl")

metadatos = {
    "version_modelo": "1.0",
    "fecha_entrenamiento": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"),
    "variables": VARIABLES,
    "k": int(K_OPTIMO),
    "silhouette": round(float(silhouettes[K_OPTIMO]), 4),
    "clientes_entrenamiento": int(len(rfm)),
    "transacciones_validas": int(len(transacciones)),
    "fecha_corte_recencia": str(fecha_corte.date()),
    "segmentos": {str(k): v for k, v in etiquetas.items()},
    "centroides": {
        str(i): dict(zip(VARIABLES, np.round(c, 3).tolist()))
        for i, c in enumerate(escalador.inverse_transform(modelo.cluster_centers_))
    },
}

with open(CARPETA_MODELO / "mapa_segmentos.json", "w", encoding="utf-8") as f:
    json.dump(metadatos, f, ensure_ascii=False, indent=2)

# Tabla de clientes con su segmento asignado, para anexar al informe
rfm.to_csv(CARPETA_SALIDA / "clientes_segmentados.csv", index=False, encoding="utf-8-sig")

print("Artefactos exportados:")
for ruta in sorted(CARPETA_SALIDA.rglob("*")):
    if ruta.is_file():
        print("  -", ruta)

## 8. Explicabilidad: cómo justificar una asignación

Función que traduce un resultado numérico en una frase comprensible. Es la que debe
alimentar el mensaje de la aplicación Streamlit.

In [ ]:
# ============================================================
# EXPLICACIÓN DE UNA ASIGNACIÓN INDIVIDUAL
# ============================================================
def explicar_cliente(recencia, frecuencia, valor_monetario, ticket_promedio):
    """Asigna un segmento y devuelve una explicación en lenguaje de negocio."""
    entrada = pd.DataFrame(
        [[recencia, frecuencia, valor_monetario, ticket_promedio]], columns=VARIABLES
    )
    entrada_esc = escalador.transform(entrada)
    cluster = int(modelo.predict(entrada_esc)[0])

    centroide = escalador.inverse_transform(modelo.cluster_centers_)[cluster]
    comparacion = []
    for variable, valor, centro in zip(VARIABLES, entrada.iloc[0], centroide):
        relacion = "por encima de" if valor >= centro else "por debajo de"
        comparacion.append(f"{variable.replace('_', ' ').lower()} {relacion} {centro:,.1f}")

    texto = (
        f"Segmento asignado: {etiquetas[cluster]}.\n"
        f"Motivo: el cliente presenta " + "; ".join(comparacion) + ".\n"
        "Este resultado describe similitud con patrones de compra observados. "
        "No debe utilizarse como única base para negar beneficios o excluir clientes."
    )
    return cluster, etiquetas[cluster], texto


# Ejemplo con el cliente de mayor valor del dataset
ejemplo = rfm.sort_values("Valor_Monetario", ascending=False).iloc[0]
_, _, mensaje = explicar_cliente(
    ejemplo.Recencia, ejemplo.Frecuencia, ejemplo.Valor_Monetario, ejemplo.Ticket_Promedio
)
print(f"Cliente de ejemplo: {ejemplo.ID_Cliente}\n")
print(mensaje)

## 9. Resumen de la ejecución

Cifras que deben coincidir con las reportadas en el documento Word.

In [ ]:
# ============================================================
# RESUMEN FINAL
# ============================================================
print("=" * 58)
print("RESUMEN DE EJECUCIÓN".center(58))
print("=" * 58)
print(f"Filas crudas                 : {len(crudo)}")
print(f"Duplicados eliminados        : {len(crudo) - len(limpio)}")
print(f"Transacciones válidas        : {len(transacciones)} "
      f"({len(transacciones) / len(crudo):.0%} del volumen original)")
print(f"Clientes modelados           : {len(rfm)}")
print(f"Clusters (k)                 : {K_OPTIMO}")
print(f"Silhouette                   : {silhouettes[K_OPTIMO]:.4f}")
print(f"Periodo analizado            : {transacciones['Fecha_Compra'].min().date()} "
      f"a {transacciones['Fecha_Compra'].max().date()}")
print("-" * 58)
for _, fila in resumen.sort_values("Valor_total", ascending=False).iterrows():
    print(f"  {fila.Segmento[:42]:<44} {int(fila.Clientes):>3} clientes "
          f"{fila.Pct_Ingresos:>6.0%} ingresos")
print("=" * 58)